# Notebook 2: Regression with PyTorch

In the previous notebook, we built a neural network from scratch. Now, we will transition to using **PyTorch**, a powerful deep learning framework.

## Key Concepts in PyTorch

1.  **Tensors**: The fundamental data structure, similar to NumPy arrays but with GPU support.
2.  **Autograd**: Automatic differentiation engine that computes gradients for us.
3.  **nn.Module**: The base class for all neural network modules.
4.  **Optimizers**: Pre-implemented optimization algorithms (SGD, Adam, etc.) in `torch.optim`.
5.  **DataLoaders**: Efficient data handling tools in `torch.utils.data`.

## Task: Redshift to Distance Regression (Revisited)

We will solve the same regression problem as before, but using PyTorch's high-level building blocks.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

# Set seed
torch.manual_seed(42)
np.random.seed(42)

# 1. Generate Data (Same as before)
def generate_data(n_samples=1000):
    z = np.random.uniform(0, 3, n_samples)
    noise = np.random.normal(0, 500, n_samples)
    D = 3000 * z + 500 * z**2 + noise
    return z.reshape(-1, 1).astype(np.float32), D.reshape(-1, 1).astype(np.float32)

X_np, y_np = generate_data()

# Normalize
X_mean, X_std = X_np.mean(), X_np.std()
y_mean, y_std = y_np.mean(), y_np.std()

X_scaled = (X_np - X_mean) / X_std
y_scaled = (y_np - y_mean) / y_std

# Convert to PyTorch Tensors
X_tensor = torch.from_numpy(X_scaled)
y_tensor = torch.from_numpy(y_scaled)

# Create DataLoader
dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f"Data shape: {X_tensor.shape}, {y_tensor.shape}")

## 2. Defining the Model with `nn.Module`

We define a class inheriting from `nn.Module`. In the `__init__`, we define the layers. In `forward`, we define how the data flows through the layers.

In [ ]:
class GalaxyNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(GalaxyNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, x):
        return self.net(x)

# Instantiate
model = GalaxyNet(input_dim=1, hidden_dim=32, output_dim=1)
print(model)

## 3. Training Loop

We use `nn.MSELoss` and `optim.Adam`. The training loop involves:
1.  Zeroing gradients (`optimizer.zero_grad()`)
2.  Forward pass (`model(x)`)
3.  Loss calculation (`criterion(pred, y)`)
4.  Backward pass (`loss.backward()`)
5.  Optimizer step (`optimizer.step()`)

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 100
loss_history = []

for epoch in range(epochs):
    epoch_loss = 0
    for batch_X, batch_y in dataloader:
        # 1. Zero gradients
        optimizer.zero_grad()
        
        # 2. Forward pass
        predictions = model(batch_X)
        
        # 3. Compute loss
        loss = criterion(predictions, batch_y)
        
        # 4. Backward pass
        loss.backward()
        
        # 5. Optimizer step
        optimizer.step()
        
        epoch_loss += loss.item()
    
    loss_history.append(epoch_loss / len(dataloader))
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss_history[-1]:.5f}")

plt.plot(loss_history)
plt.title('PyTorch Training Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.show()

In [ ]:
# Evaluation
model.eval() # Set to evaluation mode
with torch.no_grad(): # Disable gradient calculation for efficiency
    preds_scaled = model(X_tensor).numpy()

# Denormalize
preds = preds_scaled * y_std + y_mean

plt.figure(figsize=(8, 5))
plt.scatter(X_np, y_np, s=10, color='gray', alpha=0.5, label='Data')
plt.scatter(X_np, preds, s=10, color='blue', label='PyTorch Model')
plt.xlabel('Redshift (z)')
plt.ylabel('Distance (Mpc)')
plt.title('PyTorch Regression Results')
plt.legend()
plt.show()